**Importing 10-K**

In [1]:
import edgar

In [2]:
edgar.set_identity("feb2126@columbia.edu")

In [3]:
company = edgar.Company('AAPL')
company

╭───────────────────────────────────────────────────────────────────────────────────╮
│ Apple Inc.  AAPL                                                                  │
│ 0000320193 • Nasdaq • Large Accelerated Filer                                     │
│                                                                                   │
│   Industry          3571: Electronic Computers                                    │
│   Fiscal Year       Sep 26                                                        │
│   Incorporated      California                                                    │
│   Phone             (408) 996-1010                                                │
│   Address           ONE APPLE PARK WAY, CUPERTINO, CA 95014                       │
╰────────────────────────────────── SEC Entity Data • company.docs for usage guide ─╯

In [4]:
filings = company.get_filings(form="10-K")
filings

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    10-K        Annual report for public companies                            2025-10-31   0000320193-25-0000…   │
│    10-K        Annual report for public companies                            2024-11-01   0000320193-24-0001…   │
│    10-K        Annual report for public companies                            2023-11-03   0000320193-23-0001…   │
│    10-K        Annual report for public companies                     

In [5]:
latest_10k = filings.latest()


In [6]:
print([attr for attr in dir(latest_10k) if not attr.startswith('_')])

['acceptance_datetime', 'accession_no', 'accession_number', 'all_ciks', 'all_entities', 'as_company_filing', 'attachments', 'base_dir', 'cik', 'company', 'data_object', 'docs', 'document', 'exhibits', 'file_number', 'filing_date', 'filing_directory', 'filing_url', 'form', 'from_dict', 'from_json', 'from_sgml', 'from_sgml_text', 'full_text_submission', 'get_entity', 'header', 'home', 'homepage', 'homepage_url', 'html', 'index_header_url', 'index_headers', 'is_inline_xbrl', 'is_multi_entity', 'is_xbrl', 'items', 'load', 'markdown', 'obj', 'open', 'open_homepage', 'parsed_items', 'period_of_report', 'primary_doc_description', 'primary_document', 'primary_documents', 'related_filings', 'report_date', 'reports', 'save', 'search', 'sections', 'serve', 'sgml', 'size', 'statements', 'summary', 'text', 'text_url', 'to_context', 'to_dict', 'url', 'view', 'xbrl', 'xml']


In [7]:
all_reports = latest_10k.reports  # type: ignore
all_reports

╭─────────────────────────────────────────── Reports ────────────────────────────────────────────╮
│                                                                                                │
│   #    Report                                                         Category       File      │
│  ────────────────────────────────────────────────────────────────────────────────────────────  │
│   1    Cover Page                                                     Cover          R1.htm    │
│                                                                                                │
│   2    Auditor Information                                            Cover          R2.htm    │
│                                                                                                │
│   3    CONSOLIDATED STATEMENTS OF OPERATIONS                          Statements     R3.htm    │
│                                                                                                │
│   4    C

In [8]:
# Get indices and shortnames from all_reports - STATEMENTS ONLY
reports_df = all_reports.to_pandas()

# Filter to only include statements
statements_df = reports_df[reports_df['MenuCategory'] == 'Statements'].copy()

# Extract index and ShortName for statements only
indices_and_names = statements_df[['ShortName']].reset_index()
print("Statement Indices and ShortNames:")
print(indices_and_names)

# Alternative: Create a dictionary mapping for statements only
index_to_name = dict(zip(statements_df.index, statements_df['ShortName']))
print("\n" + "="*80)
print("\nStatement Index to ShortName mapping:")
for idx, name in index_to_name.items():
    print(f"  {idx}: {name}")

Statement Indices and ShortNames:
   index                                        ShortName
0      2            CONSOLIDATED STATEMENTS OF OPERATIONS
1      3  CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME
2      4                      CONSOLIDATED BALANCE SHEETS
3      5      CONSOLIDATED BALANCE SHEETS (Parenthetical)
4      6  CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY
5      7            CONSOLIDATED STATEMENTS OF CASH FLOWS


Statement Index to ShortName mapping:
  2: CONSOLIDATED STATEMENTS OF OPERATIONS
  3: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME
  4: CONSOLIDATED BALANCE SHEETS
  5: CONSOLIDATED BALANCE SHEETS (Parenthetical)
  6: CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY
  7: CONSOLIDATED STATEMENTS OF CASH FLOWS


In [9]:
# Build a dictionary to find specific statement types using keywords
import re

# Define keywords for each statement type
statement_keywords = {
    'Income Statement': ['income'],
    'Balance Sheet': ['balance sheet'],
    'Cash Flow Statement': ['cash'],
    'Equity Statement': ['equity'],
}

# Search for statements matching keywords
found_statements = {}

for statement_type, keywords in statement_keywords.items():
    for idx, row in statements_df.iterrows():
        shortname = row['ShortName'].lower()
        
        # Check if any keyword matches
        if any(keyword.lower() in shortname for keyword in keywords):
            if statement_type not in found_statements:
                found_statements[statement_type] = []
            found_statements[statement_type].append({
                'index': idx,
                'shortname': row['ShortName']
            })

# Display results
print("Found Financial Statements:")
print("="*80)
for statement_type, matches in found_statements.items():
    print(f"\n{statement_type}:")
    for match in matches:
        print(f"  Index {match['index']}: {match['shortname']}")

# Create a simplified mapping (taking first match for each type) with shortnames
statement_index_map = {}
statement_name_map = {}
for statement_type, matches in found_statements.items():
    if matches:
        statement_index_map[statement_type] = matches[0]['index']
        statement_name_map[statement_type] = matches[0]['shortname']

print("\n" + "="*80)
print("\nSimplified Statement Index Map (first match only):")
print(statement_index_map)

print("\n" + "="*80)
print("\nSimplified Statement Name Map (first match only):")
for key, value in statement_name_map.items():
    print(f"  {key}: {value}")

Found Financial Statements:

Income Statement:
  Index 3: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME

Balance Sheet:
  Index 4: CONSOLIDATED BALANCE SHEETS
  Index 5: CONSOLIDATED BALANCE SHEETS (Parenthetical)

Cash Flow Statement:
  Index 7: CONSOLIDATED STATEMENTS OF CASH FLOWS

Equity Statement:
  Index 6: CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY


Simplified Statement Index Map (first match only):
{'Income Statement': 3, 'Balance Sheet': 4, 'Cash Flow Statement': 7, 'Equity Statement': 6}


Simplified Statement Name Map (first match only):
  Income Statement: CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME
  Balance Sheet: CONSOLIDATED BALANCE SHEETS
  Cash Flow Statement: CONSOLIDATED STATEMENTS OF CASH FLOWS
  Equity Statement: CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY


In [10]:
print([attr for attr in dir(all_reports) if not attr.startswith('_')])

['create_from_record', 'current', 'data', 'data_pager', 'filter', 'get_by_category', 'get_by_filename', 'get_by_short_name', 'long_names', 'n', 'next', 'previous', 'short_names', 'statements', 'title', 'to_pandas']


In [11]:
statements = all_reports.statements
print([attr for attr in dir(statements) if not attr.startswith('_')])
detected_statments = statements.detected_statements
print([attr for attr in dir(detected_statments) if not attr.startswith('_')])
print(detected_statments.values())

['balance_sheet', 'cash_flow_statement', 'comprehensive_income_statement', 'detected_statements', 'equity_statement', 'income_statement', 'mapper', 'statements']
['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']
dict_values(['CONSOLIDATED STATEMENTS OF OPERATIONS', 'CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME', 'CONSOLIDATED BALANCE SHEETS', "CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY", 'CONSOLIDATED STATEMENTS OF CASH FLOWS'])


In [12]:
report = all_reports.get_by_filename('R50.htm') # type: ignore

In [13]:
print([attr for attr in dir(report) if not attr.startswith('_')])

['content', 'has_embedded_reports', 'html_file_name', 'instance', 'is_default', 'long_name', 'menu_category', 'parent_role', 'position', 'report_type', 'role', 'short_name', 'text', 'to_dataframe', 'view']


In [14]:
report_data = report.content
report_data

'<html>\n<head>\n<title></title>\n<link rel="stylesheet" type="text/css" href="include/report.css">\n<script type="text/javascript" src="Show.js">/* Do Not Remove This Comment */</script><script type="text/javascript">\n\t\t\t\t\t\t\tfunction toggleNextSibling (e) {\n\t\t\t\t\t\t\tif (e.nextSibling.style.display==\'none\') {\n\t\t\t\t\t\t\te.nextSibling.style.display=\'block\';\n\t\t\t\t\t\t\t} else { e.nextSibling.style.display=\'none\'; }\n\t\t\t\t\t\t\t}</script>\n</head>\n<body>\n<span style="display: none;">v3.25.3</span><table class="report" border="0" cellspacing="2" id="id2">\n<tr>\n<th class="tl" colspan="1" rowspan="2"><div style="width: 200px;"><strong>Income Taxes - Additional Information (Details)<br> $ in Millions</strong></div></th>\n<th class="th" colspan="1"></th>\n<th class="th" colspan="1">3 Months Ended</th>\n<th class="th" colspan="3">12 Months Ended</th>\n<th class="th" colspan="1"></th>\n</tr>\n<tr>\n<th class="th">\n<div>Aug. 30, 2016 </div>\n<div>Subsidiary</di

In [15]:
tenk = latest_10k.obj()
tenk

╭──────────────────────────────────────────────── Apple Inc. 10-K ────────────────────────────────────────────────╮
│ Period ending September 27, 2025 filed on October 31, 2025                                                      │
│                                                                                                                 │
│                                                                                                                 │
│ 📄                                                                                                              │
│ ├── PART I                                                                                                      │
│ │   ├── Item 1  Business                                                                                        │
│ │   ├── Item 1A Risk Factors                                                                                    │
│ │   ├── Item 1B Unresolved Staff Comments                              

In [16]:
print([attr for attr in dir(tenk) if not attr.startswith('_')])

['balance_sheet', 'business', 'cash_flow_statement', 'chunked_document', 'company', 'directors_officers_and_governance', 'doc', 'document', 'filing_date', 'financials', 'form', 'get_item_with_part', 'get_structure', 'id_parse_document', 'income_statement', 'items', 'management_discussion', 'period_of_report', 'risk_factors', 'sections', 'structure', 'view']


In [17]:
business = tenk.business
business

'Item 1.\xa0\xa0\xa0\xa0Business\n\nCompany Background\n\nThe Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Company’s fiscal year is the 52- or 53-week period that ends on the last Saturday of September.\n\nProducts\n\niPhone\n\niPhone® is the Company’s line of smartphones based on its iOS operating system. The iPhone line includes iPhone 17 Pro, iPhone Air™, iPhone 17, iPhone 16 and iPhone 16e.\n\nMac\n\nMac® is the Company’s line of personal computers based on its macOS® operating system. The Mac line includes laptops MacBook Air® and MacBook Pro®, as well as desktops iMac®, Mac mini®, Mac Studio® and Mac Pro®.\n\niPad\n\niPad® is the Company’s line of multipurpose tablets based on its iPadOS® operating system. The iPad line includes iPad Pro®, iPad Air®, iPad and iPad mini®.\n\nWearables, Home and Accessories\n\nWearables includes smartwatches, wireless headphones and spatia

**Try to make a Function that Pulls all the PARTS and Items**

In [18]:
tenk_obj = tenk.structure.structure
tenk_obj

{'PART I': {'ITEM 1': {'Title': 'Business',
   'Description': "Overview of the company's business operations, products, services, and market environment."},
  'ITEM 1A': {'Title': 'Risk Factors',
   'Description': "Discussion of risks and uncertainties that could materially affect the company's financial condition or results of operations."},
  'ITEM 1B': {'Title': 'Unresolved Staff Comments',
   'Description': "Any comments from the SEC staff on the company's previous filingsthat remain unresolved."},
  'ITEM 1C': {'Title': 'Cybersecurity',
   'Description': 'Cybersecurity risk management, strategy, and governance disclosures.'},
  'ITEM 2': {'Title': 'Properties',
   'Description': 'Information about the physical properties owned or leased by the company.'},
  'ITEM 3': {'Title': 'Legal Proceedings',
   'Description': 'Details of significant ongoing legal proceedings.'},
  'ITEM 4': {'Title': 'Mine Safety Disclosures',
   'Description': 'Relevant for mining companies, disclosures abo

In [19]:
print([attr for attr in dir(tenk_obj) if not attr.startswith('_')])

['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']


In [20]:
item_by_part = tenk.get_item_with_part(part='PART I', item='Item 1A')


In [21]:
def format_text_with_newlines(text_obj):
    """
    Format text output to properly display newlines.
    
    Args:
        text_obj: Text string or object that returns text
    
    Returns:
        None (prints formatted text)
    """
    # Get text string if it's a method/object
    if callable(text_obj):
        text = text_obj()
    else:
        text = str(text_obj)
    
    # Print the text (print() automatically renders \n as newlines)
    print(text)

In [22]:
formatted_text = format_text_with_newlines(item_by_part)

Item 1A.    Risk Factors

The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement of all potential risks or uncertainties that the Company faces or may face in the future.

This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations” and the consolidated financial statements and accompanying notes in Part II, Item 8, “Financial Statements a

In [23]:
def extract_parts_and_items(tenk_obj):
    """
    Extract PARTs and Items from a 10-K XBRL object.
    
    Args:
        tenk_obj: The 10-K object returned from filing.obj()
    
    Returns:
        dict: Dictionary with structure {PART: [Items]} and 'all_items' key for flat list
    """
    result = {}
    
    # Access the pre-built structure directly
    structure_dict = tenk_obj.structure.structure
    
    # Parse the structure dictionary
    for part, items_dict in structure_dict.items():
        result[part] = list(items_dict.keys())
        
    return result



In [24]:
tenk_dict = extract_parts_and_items(tenk)
tenk_dict

{'PART I': ['ITEM 1',
  'ITEM 1A',
  'ITEM 1B',
  'ITEM 1C',
  'ITEM 2',
  'ITEM 3',
  'ITEM 4'],
 'PART II': ['ITEM 5',
  'ITEM 6',
  'ITEM 7',
  'ITEM 7A',
  'ITEM 8',
  'ITEM 9',
  'ITEM 9A',
  'ITEM 9B',
  'ITEM 9C'],
 'PART III': ['ITEM 10', 'ITEM 11', 'ITEM 12', 'ITEM 13', 'ITEM 14'],
 'PART IV': ['ITEM 15', 'ITEM 16']}

**Function to Extract Report Data**

This function retrieves report data by filename and automatically converts statements/tables to DataFrames.

In [25]:
import pandas as pd

def get_report_data(reports_obj, filename, return_dataframe=True):
    """
    Extract data from a specific report file in a 10-K filing.
    
    Args:
        reports_obj: The reports object from filing.reports
        filename: The filename of the report (e.g., 'R3.htm', 'R5.htm')
        return_dataframe: If True and the report is a statement/table, returns DataFrame
                         If False or report is not tabular, returns the report object
    
    Returns:
        pd.DataFrame or Report object: DataFrame for statements/tables, Report object otherwise
    """
    # Get the report by filename
    report = reports_obj.get_by_filename(filename)
    
    if report is None:
        raise ValueError(f"Report with filename '{filename}' not found")
    
    # Print report info
    print(f"Report: {report.long_name}")
    print(f"Category: {report.menu_category}")
    print(f"Filename: {report.html_file_name}")
    print("-" * 80)
    
    # Check if it's a statement or table that can be converted to DataFrame
    statement_categories = 'Statements'
    
    if return_dataframe and report.menu_category in statement_categories:
        try:
            df = report.to_dataframe()
            print(f"\n✓ Converted to DataFrame: {df.shape[0]} rows × {df.shape[1]} columns\n")
            return df
        except Exception as e:
            print(f"\n✗ Could not convert to DataFrame: {e}")
            print("Returning report object instead\n")
            return report
    else:
        # For non-tabular reports (Cover, etc.), return the report object
        print(f"\nReport type '{report.menu_category}' - returning report object")
        print("Use .content or .text to access the data\n")
        return report

# Example usage
print("Available reports:")
reports_list = all_reports.to_pandas()
print(reports_list[['ShortName', 'MenuCategory', 'HtmlFileName']].head(10))

Available reports:
                                         ShortName MenuCategory HtmlFileName
0                                       Cover Page        Cover       R1.htm
1                              Auditor Information        Cover       R2.htm
2            CONSOLIDATED STATEMENTS OF OPERATIONS   Statements       R3.htm
3  CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME   Statements       R4.htm
4                      CONSOLIDATED BALANCE SHEETS   Statements       R5.htm
5      CONSOLIDATED BALANCE SHEETS (Parenthetical)   Statements       R6.htm
6  CONSOLIDATED STATEMENTS OF SHAREHOLDERS' EQUITY   Statements       R7.htm
7            CONSOLIDATED STATEMENTS OF CASH FLOWS   Statements       R8.htm
8       Summary of Significant Accounting Policies        Notes       R9.htm
9                                          Revenue        Notes      R10.htm


In [26]:
# Test 1: Get Income Statement as DataFrame
income_statement = get_report_data(all_reports, 'R3.htm')
income_statement.head()

Report: 9952151 - Statement - CONSOLIDATED STATEMENTS OF OPERATIONS
Category: Statements
Filename: R3.htm
--------------------------------------------------------------------------------

✓ Converted to DataFrame: 0 rows × 0 columns



""


In [27]:
print([attr for attr in dir(income_statement) if not attr.startswith('_')])

['T', 'abs', 'add', 'add_prefix', 'add_suffix', 'agg', 'aggregate', 'align', 'all', 'any', 'apply', 'applymap', 'asfreq', 'asof', 'assign', 'astype', 'at', 'at_time', 'attrs', 'axes', 'backfill', 'between_time', 'bfill', 'bool', 'boxplot', 'clip', 'columns', 'combine', 'combine_first', 'compare', 'convert_dtypes', 'copy', 'corr', 'corrwith', 'count', 'cov', 'cummax', 'cummin', 'cumprod', 'cumsum', 'describe', 'diff', 'div', 'divide', 'dot', 'drop', 'drop_duplicates', 'droplevel', 'dropna', 'dtypes', 'duplicated', 'empty', 'eq', 'equals', 'eval', 'ewm', 'expanding', 'explode', 'ffill', 'fillna', 'filter', 'first', 'first_valid_index', 'flags', 'floordiv', 'from_dict', 'from_records', 'ge', 'get', 'groupby', 'gt', 'head', 'hist', 'iat', 'idxmax', 'idxmin', 'iloc', 'index', 'infer_objects', 'info', 'insert', 'interpolate', 'isetitem', 'isin', 'isna', 'isnull', 'items', 'iterrows', 'itertuples', 'join', 'keys', 'kurt', 'kurtosis', 'last', 'last_valid_index', 'le', 'loc', 'lt', 'map', 'mask

In [28]:
# Test 2: Get Balance Sheet as DataFrame
balance_sheet = get_report_data(all_reports, 'R5.htm')
balance_sheet

Report: 9952153 - Statement - CONSOLIDATED BALANCE SHEETS
Category: Statements
Filename: R5.htm
--------------------------------------------------------------------------------

✓ Converted to DataFrame: 0 rows × 0 columns



""


In [29]:
# Test 3: Get Notes report - it will be converted to DataFrame by default
notes_report = get_report_data(all_reports, 'R50.htm')

print("\n✓ Result is a Report object")
# Access the text property (not a method)
print("\n" + "="*80)
print("REPORT TEXT DATA:")
print("="*80)
print(notes_report.text())  # Access text as a property, not a method

Report: 9955536 - Disclosure - Income Taxes - Additional Information (Details)
Category: Details
Filename: R50.htm
--------------------------------------------------------------------------------

Report type 'Details' - returning report object
Use .content or .text to access the data


✓ Result is a Report object

REPORT TEXT DATA:


                                                                                                                   
 Income Taxes - Additional                                                                                         
 Information (Details)                                                                                             
 $ in Millions                                        3 Months                                                     
 Income Taxes - Additional                               Ended     12 Months                                       
 Information (Details)                   Aug. 30,     Sep. 28,         Ended     Sep. 28,    Sep. 30,     Sep. 24, 
 $ in Millions                               2016         2024 Sep. 27, 2025         2024        2023         2022 
 Income Tax Contingency [Line Items]   Subsidiary      USD ($)       USD ($)      USD ($)     USD ($)      USD ($) 
 ───────────────────────────────────────────────────────────────────────

**Write a Funtion to give me a list of string with the file names**

In [30]:
reports_df = all_reports.to_pandas()
reports_df

,instance,IsDefault,HasEmbeddedReports,HtmlFileName,LongName,ReportType,Role,ParentRole,ShortName,MenuCategory,Position
0,aapl-20250927.htm,False,False,R1.htm,0000001 - Document - Cover Page,Sheet,http://www.apple.com/role/CoverPage,None,Cover Page,Cover,1
1,aapl-20250927.htm,False,False,R2.htm,0000002 - Document - Auditor Information,Sheet,http://www.apple.com/role/AuditorInformation,None,Auditor Information,Cover,2
2,aapl-20250927.htm,False,False,R3.htm,9952151 - Statement - CONSOLIDATED STATEMENTS ...,Sheet,http://www.apple.com/role/CONSOLIDATEDSTATEMEN...,None,CONSOLIDATED STATEMENTS OF OPERATIONS,Statements,3
3,aapl-20250927.htm,False,False,R4.htm,9952152 - Statement - CONSOLIDATED STATEMENTS ...,Sheet,http://www.apple.com/role/CONSOLIDATEDSTATEMEN...,None,CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME,Statements,4
4,aapl-20250927.htm,False,False,R5.htm,9952153 - Statement - CONSOLIDATED BALANCE SHEETS,Sheet,http://www.apple.com/role/CONSOLIDATEDBALANCES...,None,CONSOLIDATED BALANCE SHEETS,Statements,5
...,...,...,...,...,...,...,...,...,...,...,...
66,aapl-20250927.htm,False,False,R67.htm,"9955553 - Disclosure - Commitments, Contingenc...",Sheet,http://www.apple.com/role/CommitmentsContingen...,None,"Commitments, Contingencies and Supply Concentr...",Details,67
67,aapl-20250927.htm,False,False,R68.htm,9955554 - Disclosure - Segment Information and...,Sheet,http://www.apple.com/role/SegmentInformationan...,None,Segment Information and Geographic Data - Info...,Details,68
68,aapl-20250927.htm,False,False,R69.htm,9955555 - Disclosure - Segment Information and...,Sheet,http://www.apple.com/role/SegmentInformationan...,None,Segment Information and Geographic Data - Net ...,Details,69
69,aapl-20250927.htm,False,False,R70.htm,9955556 - Disclosure - Segment Information and...,Sheet,http://www.apple.com/role/SegmentInformationan...,None,Segment Information and Geographic Data - Long...,Details,70


In [31]:
def get_report_filenames(reports_obj):
    """
    Extract all report filenames from a reports object.
    
    Args:
        reports_obj: The reports object from filing.reports
    
    Returns:
        list: List of filename strings (e.g., ['R1.htm', 'R2.htm', ...])
    """
    # Convert reports to DataFrame and extract HtmlFileName column
    reports_df = reports_obj.to_pandas()
    filenames = reports_df['HtmlFileName'].dropna().tolist()  # Drop None values
    
    return filenames


In [32]:
filenames = get_report_filenames(all_reports)
print(f"Found {len(filenames)} report files:\n")
print(filenames)

Found 70 report files:

['R1.htm', 'R2.htm', 'R3.htm', 'R4.htm', 'R5.htm', 'R6.htm', 'R7.htm', 'R8.htm', 'R9.htm', 'R10.htm', 'R11.htm', 'R12.htm', 'R13.htm', 'R14.htm', 'R15.htm', 'R16.htm', 'R17.htm', 'R18.htm', 'R19.htm', 'R20.htm', 'R21.htm', 'R22.htm', 'R23.htm', 'R24.htm', 'R25.htm', 'R26.htm', 'R27.htm', 'R28.htm', 'R29.htm', 'R30.htm', 'R31.htm', 'R32.htm', 'R33.htm', 'R34.htm', 'R35.htm', 'R36.htm', 'R37.htm', 'R38.htm', 'R39.htm', 'R40.htm', 'R41.htm', 'R42.htm', 'R43.htm', 'R44.htm', 'R45.htm', 'R46.htm', 'R47.htm', 'R48.htm', 'R49.htm', 'R50.htm', 'R51.htm', 'R52.htm', 'R53.htm', 'R54.htm', 'R55.htm', 'R56.htm', 'R57.htm', 'R58.htm', 'R59.htm', 'R60.htm', 'R61.htm', 'R62.htm', 'R63.htm', 'R64.htm', 'R65.htm', 'R66.htm', 'R67.htm', 'R68.htm', 'R69.htm', 'R70.htm']


**Pulling the statment files Directly**

In [33]:
xb = latest_10k.xbrl()
xb

╭───────────────────────────────────────────────── XBRL Document ─────────────────────────────────────────────────╮
│ Apple Inc. (AAPL) • CIK 0000320193                                                                              │
│                                                                                                                 │
│          Form:  10-K                                                                                            │
│ Fiscal Period:  Fiscal Year 2025 (ended Sep 27, 2025)                                                           │
│          Data:  1,131 facts • 182 contexts                                                                      │
│                                                                                                                 │
│ Periods Available for Statements:                                                                               │
│   Annual: FY 2025, FY 2024, FY 2023                                   

In [34]:
statements = xb.statements
statements

Financial Statements                                                                         
                                                                                             
  #     Name                                           Type                  Parenthetical   
 ─────────────────────────────────────────────────────────────────────────────────────────── 
  4     CONSOLIDATEDBALANCESHEETS                      BalanceSheet                          
  7     CONSOLIDATEDSTATEMENTSOFCASHFLOWS              CashFlowStatement                     
  3     CONSOLIDATEDSTATEMENTSOFCOMPREHENSIVEINCOME    ComprehensiveIncome                   
  2     CONSOLIDATEDSTATEMENTSOFOPERATIONS             IncomeStatement                       
  6     CONSOLIDATEDSTATEMENTSOFSHAREHOLDERSEQUITY     StatementOfEquity                     
  59    ShareholdersEquitySharesofCommonStockDetails   StatementOfEquity                     
                                                            

In [35]:
print([attr for attr in dir(statements) if not attr.startswith('_')])

['balance_sheet', 'cashflow_statement', 'classify_statement', 'comprehensive_income', 'cover_page', 'disclosures', 'find_statement_by_primary_concept', 'get_by_category', 'get_period_views', 'get_statements_by_category', 'income_statement', 'notes', 'schedule_of_investments', 'statement_by_type', 'statement_of_equity', 'statements', 'to_dataframe', 'xbrl']


In [36]:
stmt = statements[5]
print([attr for attr in dir(stmt) if not attr.startswith('_')])
stmt_df = stmt
stmt_df



No periods with sufficient data found for Apple Inc. BalanceSheetParenthetical, returning all candidates


['REQUIRED_CONCEPTS', 'analyze_trends', 'calculate_ratios', 'canonical_type', 'docs', 'get_raw_data', 'is_segmented', 'primary_concept', 'render', 'role_or_type', 'to_dataframe', 'xbrl']


                             CONSOLIDATEDBALANCESHEETSParenthetical                              
                                           Year Ended                                            
                                                                                                 
                                                     Sep 27, 2025   Sep 28, 2024   Sep 30, 2023  
 ─────────────────────────────────────────────────────────────────────────────────────────────── 
    Common stock, par value (in dollars per share)                                               
    Common stock, shares authorized (in shares)                                                  
    Common Stock Shares Issued                                                                   
    Common Stock Shares Outstanding:                                                             
                                                                                                 
              Apple 